In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [ ]:
# -----------------------------
# 1. Load dataset
# -----------------------------
df = pd.read_excel("data1319.xlsx")

In [ ]:
# -----------------------------
# 2. Temporal split
# -----------------------------
train_df = df[df['year'] <= 2018].copy()
test_df  = df[df['year'] == 2019].copy()

In [ ]:
y_train = train_df['share']
y_test  = test_df['share']

In [ ]:
exclude_cols = ["share", "mo", "year", "model"]
X_train = train_df.drop(columns=exclude_cols)
X_test  = test_df.drop(columns=exclude_cols)

In [ ]:
# -----------------------------
# 3. Encoding (no leakage)
# -----------------------------
for col in X_train.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col]  = le.transform(X_test[col])

In [ ]:
# -----------------------------
# 4. TimeSeries CV
# -----------------------------
tscv = TimeSeriesSplit(n_splits=4)

In [ ]:
# -----------------------------
# 5. XGBOOST (Improved CV)
# -----------------------------
xgb_model = XGBRegressor(random_state=42)

In [ ]:
xgb_param_grid = {
    "n_estimators": [300, 500],
    "max_depth": [3, 5],
    "learning_rate": [0.01, 0.05],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

In [ ]:
xgb_grid = GridSearchCV(
    xgb_model,
    xgb_param_grid,
    cv=tscv,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

In [ ]:
xgb_grid.fit(X_train, y_train)
xgb_best = xgb_grid.best_estimator_

In [ ]:
print("\nBest XGBoost Params:", xgb_grid.best_params_)

In [ ]:
# -----------------------------
# 6. RANDOM FOREST (NEW)
# -----------------------------
rf_model = RandomForestRegressor(random_state=42)

In [ ]:
rf_param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

In [ ]:
rf_grid = GridSearchCV(
    rf_model,
    rf_param_grid,
    cv=tscv,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

In [ ]:
rf_grid.fit(X_train, y_train)
rf_best = rf_grid.best_estimator_

In [ ]:
print("\nBest Random Forest Params:", rf_grid.best_params_)

In [ ]:
# -----------------------------
# 7. Evaluation function
# -----------------------------
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)
    
    return {
        "Model": name,
        "Train R2": r2_score(y_train, y_train_pred),
        "Test R2": r2_score(y_test, y_test_pred),
        "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
    }

In [ ]:
# -----------------------------
# 8. Compare models
# -----------------------------
results = []

In [ ]:
results.append(evaluate_model("XGBoost", xgb_best, X_train, X_test, y_train, y_test))
results.append(evaluate_model("Random Forest", rf_best, X_train, X_test, y_train, y_test))

In [ ]:
results_df = pd.DataFrame(results)
print("\nModel Comparison:")
print(results_df.sort_values(by="Test R2", ascending=False))